**Import Required Libraries**

In [53]:
import torch
import torch.nn as nn
import math

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.8.0+cu126
CUDA available: False


**Scaled Dot-Product Attention**

This function implements the fundamental Attention mechanism.

- Formula:
    - Attention(Q, K, V) = softmax( (QKᵀ) / √dₖ ) × V


In [54]:
class ScaledDotProductAttention(nn.Module):
    """
    Scaled Dot-Product Attention mechanism.

    Computes: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    """
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: Query tensor (batch_size, num_heads, seq_len_q, d_k)
            K: Key tensor (batch_size, num_heads, seq_len_k, d_k)
            V: Value tensor (batch_size, num_heads, seq_len_v, d_v)
            mask: Optional mask tensor

        Returns:
            attention_output: (batch_size, num_heads, seq_len_q, d_v)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k)
        """
        d_k = Q.size(-1)

        # Compute attention scores: QK^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        # Apply mask if provided (set masked positions to large negative value)
        if mask is not None:
            # Expand mask to match scores dimensions if needed
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)  # (batch_size, 1, seq_len, seq_len)
            scores = scores.masked_fill(mask == False, -1e9)

        # Apply softmax to get attention weights
        attention_weights = self.softmax(scores)

        # Multiply by values
        attention_output = torch.matmul(attention_weights, V)

        return attention_output, attention_weights

print("✓ ScaledDotProductAttention class defined successfully")

✓ ScaledDotProductAttention class defined successfully


**Multi-Head Attention**

In [55]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism.

    Projects Q, K, V to h heads, applies attention, concatenates and projects.
    """
    def __init__(self, d_model, num_heads):
        """
        Args:
            d_model: Model dimension (512)
            num_heads: Number of attention heads (8)
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 64

        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Attention mechanism
        self.attention = ScaledDotProductAttention()

        # Final linear projection
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: Query tensor (batch_size, seq_len_q, d_model)
            K: Key tensor (batch_size, seq_len_k, d_model)
            V: Value tensor (batch_size, seq_len_v, d_model)
            mask: Optional mask tensor

        Returns:
            output: (batch_size, seq_len_q, d_model)
        """
        batch_size = Q.size(0)

        # Linear projections
        Q = self.W_q(Q)  # (batch_size, seq_len_q, d_model)
        K = self.W_k(K)  # (batch_size, seq_len_k, d_model)
        V = self.W_v(V)  # (batch_size, seq_len_v, d_model)

        # Split into multiple heads: (batch_size, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Apply attention
        attention_output, attention_weights = self.attention(Q, K, V, mask)

        # Concatenate heads: (batch_size, seq_len_q, d_model)
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )

        # Final linear projection
        output = self.W_o(attention_output)

        return output

print("✓ MultiHeadAttention class defined successfully")

✓ MultiHeadAttention class defined successfully


**Position-wise Feed-Forward Network**

In [56]:
class PositionWiseFeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.

    FFN(x) = max(0, xW1 + b1)W2 + b2
    Two linear layers with ReLU activation.
    """
    def __init__(self, d_model, d_ff):
        """
        Args:
            d_model: Model dimension (512)
            d_ff: Feed-forward inner dimension (2048)
        """
        super(PositionWiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        """
        Args:
            x: Input tensor (batch_size, seq_len, d_model)

        Returns:
            output: (batch_size, seq_len, d_model)
        """
        return self.linear2(self.relu(self.linear1(x)))

print("✓ PositionWiseFeedForward class defined successfully")

✓ PositionWiseFeedForward class defined successfully


**Positional Encoding**

In [57]:
class PositionalEncoding(nn.Module):
    """
    Positional Encoding using sine and cosine functions.

    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=5000):
        """
        Args:
            d_model: Model dimension (512)
            max_len: Maximum sequence length
        """
        super(PositionalEncoding, self).__init__()

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                            (-math.log(10000.0) / d_model))

        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term)
        # Apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: Input embeddings (batch_size, seq_len, d_model)

        Returns:
            output: Input embeddings + positional encoding
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return x

print("✓ PositionalEncoding class defined successfully")

# Test positional encoding
pos_enc = PositionalEncoding(d_model=512)
sample_input = torch.randn(2, 10, 512)
output = pos_enc(sample_input)
print(f"  Input shape: {sample_input.shape}")
print(f"  Output shape: {output.shape}")

✓ PositionalEncoding class defined successfully
  Input shape: torch.Size([2, 10, 512])
  Output shape: torch.Size([2, 10, 512])


**Encoder Layer**

In [58]:
class EncoderLayer(nn.Module):
    """
    Single Encoder Layer.

    Contains:
    1. Multi-Head Self-Attention
    2. Add & Norm (Residual Connection + Layer Normalization)
    3. Position-wise Feed-Forward Network
    4. Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout):
        """
        Args:
            d_model: Model dimension (512)
            num_heads: Number of attention heads (8)
            d_ff: Feed-forward inner dimension (2048)
            dropout: Dropout rate (0.1)
        """
        super(EncoderLayer, self).__init__()

        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: Input tensor (batch_size, seq_len, d_model)
            mask: Optional padding mask

        Returns:
            output: (batch_size, seq_len, d_model)
        """
        # Multi-Head Self-Attention with residual connection and layer norm
        attention_output = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attention_output))

        # Feed-Forward Network with residual connection and layer norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

print("✓ EncoderLayer class defined successfully")

✓ EncoderLayer class defined successfully


**Decoder Layer**

In [59]:
class DecoderLayer(nn.Module):
    """
    Single Decoder Layer.

    Contains:
    1. Masked Multi-Head Self-Attention
    2. Add & Norm
    3. Multi-Head Cross-Attention (to encoder output)
    4. Add & Norm
    5. Position-wise Feed-Forward Network
    6. Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout):
        """
        Args:
            d_model: Model dimension (512)
            num_heads: Number of attention heads (8)
            d_ff: Feed-forward inner dimension (2048)
            dropout: Dropout rate (0.1)
        """
        super(DecoderLayer, self).__init__()

        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        Args:
            x: Decoder input (batch_size, tgt_seq_len, d_model)
            encoder_output: Encoder output (batch_size, src_seq_len, d_model)
            src_mask: Source padding mask
            tgt_mask: Target mask (padding + look-ahead)

        Returns:
            output: (batch_size, tgt_seq_len, d_model)
        """
        # Masked Multi-Head Self-Attention with residual and layer norm
        self_attention_output = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attention_output))

        # Multi-Head Cross-Attention with residual and layer norm
        cross_attention_output = self.cross_attention(x, encoder_output,
                                                      encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attention_output))

        # Feed-Forward Network with residual and layer norm
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

print("✓ DecoderLayer class defined successfully")

✓ DecoderLayer class defined successfully


**Complete Transformer Model**

In [60]:
class Transformer(nn.Module):
    """
    Complete Transformer Model.

    Stacks N=6 Encoder layers and N=6 Decoder layers.
    Includes embedding layers and final output projection.
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8,
                 num_layers=6, d_ff=2048, dropout=0.1, max_len=5000):
        """
        Args:
            src_vocab_size: Source vocabulary size
            tgt_vocab_size: Target vocabulary size
            d_model: Model dimension (512)
            num_heads: Number of attention heads (8)
            num_layers: Number of encoder/decoder layers (6)
            d_ff: Feed-forward inner dimension (2048)
            dropout: Dropout rate (0.1)
            max_len: Maximum sequence length
        """
        super(Transformer, self).__init__()

        self.d_model = d_model

        # Embedding layers
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding
        self.positional_encoding = PositionalEncoding(d_model, max_len)

        # Encoder stack (N=6 layers)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Decoder stack (N=6 layers)
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Final output projection
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)

    def generate_padding_mask(self, seq, pad_idx=0):
        """
        Generate padding mask to ignore <PAD> tokens.

        Args:
            seq: Input sequence (batch_size, seq_len)
            pad_idx: Padding token index

        Returns:
            mask: (batch_size, 1, seq_len)
        """
        mask = (seq != pad_idx).unsqueeze(1)
        return mask

    def generate_look_ahead_mask(self, size):
        """
        Generate look-ahead (causal) mask to prevent attention to future tokens.

        Args:
            size: Sequence length

        Returns:
            mask: (size, size)
        """
        mask = torch.tril(torch.ones((size, size)))
        return mask

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        Args:
            src: Source sequence (batch_size, src_seq_len)
            tgt: Target sequence (batch_size, tgt_seq_len)
            src_mask: Source padding mask
            tgt_mask: Target mask (padding + look-ahead)

        Returns:
            output: (batch_size, tgt_seq_len, tgt_vocab_size)
        """
        # Source embedding + positional encoding
        src_embedded = self.src_embedding(src) * math.sqrt(self.d_model)
        src_embedded = self.positional_encoding(src_embedded)
        src_embedded = self.dropout(src_embedded)

        # Pass through encoder stack
        encoder_output = src_embedded
        for encoder_layer in self.encoder_layers:
            encoder_output = encoder_layer(encoder_output, src_mask)

        # Target embedding + positional encoding
        tgt_embedded = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        tgt_embedded = self.positional_encoding(tgt_embedded)
        tgt_embedded = self.dropout(tgt_embedded)

        # Pass through decoder stack
        decoder_output = tgt_embedded
        for decoder_layer in self.decoder_layers:
            decoder_output = decoder_layer(decoder_output, encoder_output,
                                          src_mask, tgt_mask)

        # Final output projection
        output = self.output_projection(decoder_output)

        return output

print("✓ Transformer class defined successfully")
print("\n" + "="*70)
print("ALL COMPONENTS SUCCESSFULLY DEFINED!")
print("="*70)

✓ Transformer class defined successfully

ALL COMPONENTS SUCCESSFULLY DEFINED!


**Model Instantiation and Testing**

In [61]:
print("\n" + "="*70)
print("TRANSFORMER MODEL - BASIC FUNCTIONALITY TEST")
print("="*70)

# Hyperparameters (Base Model Configuration)
src_vocab_size = 10000
tgt_vocab_size = 10000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
dropout = 0.1

print("\n📋 Model Hyperparameters:")
print(f"  • d_model: {d_model}")
print(f"  • num_heads: {num_heads}")
print(f"  • num_layers: {num_layers}")
print(f"  • d_k = d_v: {d_model // num_heads}")
print(f"  • d_ff: {d_ff}")
print(f"  • dropout: {dropout}")

# Instantiate model
print("\n" + "-"*70)
print("🔧 Instantiating Transformer model...")
model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    dropout=dropout
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"  ✓ Model instantiated successfully!")
print(f"  ✓ Total parameters: {total_params:,}")


TRANSFORMER MODEL - BASIC FUNCTIONALITY TEST

📋 Model Hyperparameters:
  • d_model: 512
  • num_heads: 8
  • num_layers: 6
  • d_k = d_v: 64
  • d_ff: 2048
  • dropout: 0.1

----------------------------------------------------------------------
🔧 Instantiating Transformer model...
  ✓ Model instantiated successfully!
  ✓ Total parameters: 59,508,496


**Forward Pass with Dummy Data**

In [62]:
# Create dummy input tensors
print("\n" + "-"*70)
print("📊 Creating dummy input tensors...")
batch_size = 2
src_seq_len = 10
tgt_seq_len = 12

src = torch.randint(1, src_vocab_size, (batch_size, src_seq_len))
tgt = torch.randint(1, tgt_vocab_size, (batch_size, tgt_seq_len))

print(f"  • Source shape: {src.shape} (batch_size, src_seq_len)")
print(f"  • Target shape: {tgt.shape} (batch_size, tgt_seq_len)")

# Generate masks
print("\n" + "-"*70)
print("🎭 Generating masks...")
src_mask = model.generate_padding_mask(src)
tgt_padding_mask = model.generate_padding_mask(tgt)
tgt_look_ahead_mask = model.generate_look_ahead_mask(tgt_seq_len).to(tgt.device)

# Combine target masks
tgt_mask = tgt_padding_mask.unsqueeze(2).bool() & tgt_look_ahead_mask.unsqueeze(0).bool()


print(f"  • Source padding mask shape: {src_mask.shape}")
print(f"  • Target look-ahead mask shape: {tgt_look_ahead_mask.shape}")
print(f"  • Combined target mask shape: {tgt_mask.shape}")

# Forward pass
print("\n" + "-"*70)
print("⚡ Performing forward pass...")
model.eval()
with torch.no_grad():
    output = model(src, tgt, src_mask, tgt_mask)

print(f"  ✓ Forward pass completed!")
print(f"  • Output shape: {output.shape}")
print(f"    Expected: (batch_size={batch_size}, tgt_seq_len={tgt_seq_len}, tgt_vocab_size={tgt_vocab_size})")


----------------------------------------------------------------------
📊 Creating dummy input tensors...
  • Source shape: torch.Size([2, 10]) (batch_size, src_seq_len)
  • Target shape: torch.Size([2, 12]) (batch_size, tgt_seq_len)

----------------------------------------------------------------------
🎭 Generating masks...
  • Source padding mask shape: torch.Size([2, 1, 10])
  • Target look-ahead mask shape: torch.Size([12, 12])
  • Combined target mask shape: torch.Size([2, 1, 12, 12])

----------------------------------------------------------------------
⚡ Performing forward pass...
  ✓ Forward pass completed!
  • Output shape: torch.Size([2, 12, 10000])
    Expected: (batch_size=2, tgt_seq_len=12, tgt_vocab_size=10000)


**Dimension Verification**

In [63]:
print("\n" + "-"*70)
print("✅ DIMENSION VERIFICATION")
print("-"*70)

expected_shape = (batch_size, tgt_seq_len, tgt_vocab_size)
if output.shape == expected_shape:
    print(f"✓ Output shape is CORRECT: {output.shape}")
    print(f"  • Batch size: {output.shape[0]}")
    print(f"  • Target sequence length: {output.shape[1]}")
    print(f"  • Target vocabulary size: {output.shape[2]}")
else:
    print(f"✗ Output shape mismatch!")
    print(f"  Expected: {expected_shape}")
    print(f"  Got: {output.shape}")

print("\n" + "="*70)
print("TEST COMPLETED SUCCESSFULLY! ✨")
print("="*70)


----------------------------------------------------------------------
✅ DIMENSION VERIFICATION
----------------------------------------------------------------------
✓ Output shape is CORRECT: torch.Size([2, 12, 10000])
  • Batch size: 2
  • Target sequence length: 12
  • Target vocabulary size: 10000

TEST COMPLETED SUCCESSFULLY! ✨
